In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import LeaveOneGroupOut
import warnings
warnings.filterwarnings('ignore')

class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=4, n_times=7500,
                 F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1*D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1, 16), padding=(0, 8),
                      groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )
        fc_size = F2 * (n_times // 32)
        self.classifier = nn.Linear(fc_size, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

class TimeSeriesDataset(Dataset):
    def __init__(self, data_dir, labels_df):
        self.samples, self.labels = [], []
        for _, row in labels_df.iterrows():
            fname = f"sub{int(row['subject']):03d}_ses{int(row['song_id']):02d}.npy"
            fpath = os.path.join(data_dir, fname)
            if not os.path.exists(fpath):
                continue
            self.samples.append(np.load(fpath).astype(np.float32))
            self.labels.append(int(row['enjoyment']) - 1)

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x = torch.tensor(self.samples[idx]).unsqueeze(0)  # (1, 4, 7500)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

def run():
    data_root = r"C:\Users\hiro2\Documents\EEG_project\data"
    ts_dir    = os.path.join(data_root, "timeseries")
    labels_df = pd.read_csv(os.path.join(data_root, "features_clean.csv"))
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[EEGNet] device: {device}")

    dataset = TimeSeriesDataset(ts_dir, labels_df)
    print(f"[EEGNet] データ数: {len(dataset)}")

    x0, _ = dataset[0]
    n_times = x0.shape[-1]
    print(f"[EEGNet] 入力shape: {x0.shape}")

    groups = labels_df['subject'].values[:len(dataset)]
    logo   = LeaveOneGroupOut()
    all_accs = []

    for fold, (tr_idx, te_idx) in enumerate(
            logo.split(range(len(dataset)),
                       [dataset[i][1].item() for i in range(len(dataset))],
                       groups)):
        tr_loader = DataLoader(
            torch.utils.data.Subset(dataset, tr_idx),
            batch_size=8, shuffle=True)
        te_loader = DataLoader(
            torch.utils.data.Subset(dataset, te_idx),
            batch_size=8)

        model     = EEGNet(n_classes=5, n_channels=4,
                           n_times=n_times).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(50):
            model.train()
            for x, y in tr_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for x, y in te_loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
        acc = correct / len(te_idx)
        all_accs.append(acc)
        print(f"[EEGNet] Fold {fold+1:2d}: acc={acc:.3f}")

    print(f"\n[EEGNet] 平均acc: {np.mean(all_accs):.3f} ± {np.std(all_accs):.3f}")
    print(f"[EEGNet] チャンス精度: {1/5:.3f}")

if __name__ == '__main__':
    run()